<a href="https://colab.research.google.com/github/kediwudiko-commits/Keddy/blob/main/EMSC2010_Precipitation%20Pirates_Week%208.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMSC2010 Group Project 8

## 2. Roles and contributions

| Role | Primary | Deputy | Completed? | Notes |
| :--- | :--- | :--- | :--- | :--- |
| Github & integration | Kedi | Lee | `Yes`/Partial/No| Add note|
| Data steward | Lee | Khai | `Yes`/Partial/No| Add note |
| Analysis / modelling | Khai | Juliet | `Yes`/Partial/No| Generated an initial visualisation of the data. Conducted a linear regression. Generated a plot of rho.|
| Visualisation / interpretation | Juliet | Binyao | `Yes`/Partial/No| Add note|
| Narrative | Binyao | Kedi | `Yes`/Partial/No| Add note|
| Quality Control / Reproducibility  | Name | Name | Yes/Partial/No/NA| Add note|


## 1. Project Overview
Group name: Precipitation Pirates

Project week: 8

Project title: Antartic warming and Artic warming comparision (covarience and regression)

Datasets used ([ZonAnn.TS+dSST.csv](https://data.giss.nasa.gov/gistemp/tabledata_v4/ZonAnn.Ts+dSST.csv) The data was sourced from: https://svs.gsfc.nasa.gov/4978/):

## 3. Deputy Interventions (if applicable)
Repeat text as required.

* Role affected:

* Reason (e.g. missed deadline, absence, etc.):

* Deputy action taken:

* Impact on workflow:

*N.B., this section should be factual, not judgemental.*

## 4. Pre-submission checklist
* Notebook runs from top to bottom.
* Datafiles are included in the GitHub repository.
* Commits include meaningful information.
* Each group member has included a brief reflection in the notebook.
* Repository has been shared with the teaching team once your project is completed.

# Start your group project here

This project focuses on a critical question in global climate change: Is there synchronicity in the warming of Earth's poles?. Utilizing NASA’s datasets on Arctic and Antarctic temperature anomalies, we conduct a deep dive into their correlation through Bayesian statistical modeling. Moving beyond traditional point estimates, we utilize the PyMC framework to construct a bivariate normal model, allowing us to derive the posterior mean of the correlation coefficient while quantifying predictive uncertainty to reveal the complex interconnectedness of polar climate systems under global warming.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import arviz as az

In [ ]:
df = pd.read_csv("ZonAnn.Ts+dSST.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Extract relevant columns using the correct names
df_clean = df[['Year', '64N-90N', '90S-64S']].copy()

# Rename for clarity
df_clean.columns = ['Year', 'Arctic_Temp', 'Antarctic_Temp']

# Drop missing values
df_clean = df_clean.dropna()

print(df_clean.head())

The dataset used in this analysis provides temperature anomalies grouped by latitude bands rather than a single global average.

This allows for direct extraction of Arctic (90°N–64°N) and Antarctic (64°S–90°S) temperature trends without requiring additional geographic processing.

Using zonal data ensures that both regions are derived from the same dataset and methodology, improving consistency and reliability in the analysis.

## Data Cleaning and Quality Control

To prepare the data for analysis, the following cleaning steps were performed:

1.  **Column Name Cleaning:** Whitespace was stripped from all column names to ensure consistency and prevent errors during column selection.
2.  **Relevant Column Extraction:** Only the 'Year', '64N-90N' (Arctic), and '90S-64S' (Antarctic) columns were selected, as these are the primary focus of the analysis.
3.  **Column Renaming:** The extracted regional temperature columns were renamed to 'Arctic_Temp' and 'Antarctic_Temp' for better readability and understanding.
4.  **Handling Missing Values:** Any rows containing missing values (`NaN`) in the selected columns were removed to ensure data integrity for subsequent analysis.

The data was sourced from: https://svs.gsfc.nasa.gov/4978/

During the data processing phase, we first standardized the Arctic and Antarctic temperature anomaly data to eliminate scaling effects and accelerate model convergence. For model construction, we adopted a "simple-to-complex" strategy, utilizing the bambi library to fit four different orders of models, ranging from linear to quartic polynomials. To objectively evaluate the explanatory power of each model, we introduced a model comparison mechanism based on Leave-One-Out Cross-Validation (LOO-CV). The results indicate that higher-order polynomial models—specifically the Quartic Model—perform best in balancing goodness-of-fit with generalization capability, capturing subtle non-linear features in the relationship between polar temperatures.

In [ ]:
# Here we generate an initial visualisation of the data

plt.figure(figsize=(10,5))
plt.plot(df_clean["Year"], df_clean["Arctic_Temp"], label="Arctic (64N–90N)", color="firebrick")
plt.plot(df_clean["Year"], df_clean["Antarctic_Temp"], label="Antarctic (90S–64S)", color="royalblue")

plt.title("Arctic vs Antarctic Temperature Anomalies")
plt.xlabel("Year")
plt.ylabel("Temperature Anomaly (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The initial time-series plot reveals that both Arctic and Antarctic temperature anomalies exhibit an overall upward trend from 1880 to 2020. The Arctic (red line) shows more pronounced fluctuations and a steeper rise, particularly after 1980, while the Antarctic (blue line) displays a more gradual increase, suggesting differences in the rate and magnitude of warming between the two polar regions.

In [ ]:
arctic = df_clean["Arctic_Temp"]
antarctic = df_clean["Antarctic_Temp"]

# Calculate deviations from the mean
x = arctic - np.mean(arctic)
y = antarctic - np.mean(antarctic)

plt.figure(figsize=(8,6))
plt.plot(x,y,'ok') #plot the deviations
plt.gca().axvline(x=0, color = 'k', linestyle = '--') #add a vertical line at x=0
plt.gca().axhline(y=0, color = 'k', linestyle = '--') #add a horizontal line at y=0
plt.xlabel('Arctic Temperature Deviation (°C)') #label the x-axis
plt.ylabel('Antarctic Temperature Deviation (°C)') #label the y-axis
plt.title('Deviations from Mean: Arctic vs Antarctic Temperature Anomalies')
plt.minorticks_on() #include minor ticks
plt.grid(alpha=0.3)
plt.show()

The deviation plot shows how temperature anomalies in the Arctic and Antarctic vary relative to their respective means. While there is a slight tendency for both regions to move in the same direction (points loosely aligned along a positive trend), the scatter is widely dispersed across all quadrants. This indicates a weak relationship between Arctic and Antarctic temperature variability. Periods of warming or cooling in one region do not consistently correspond to similar behaviour in the other, suggesting that the two systems are only loosely coupled.

In [ ]:
# Calculate the correlation coefficient r

cov_matrix = np.cov(arctic, antarctic)
covariance = cov_matrix[0, 1]

print("Covariance:", covariance)

correlation = np.corrcoef(arctic, antarctic)[0, 1]
print("Correlation:", correlation)

The calculated correlation coefficient (r ≈ 0.23) indicates a weak positive linear relationship between Arctic and Antarctic temperature anomalies.

In [ ]:
# Perform a linear regression on the data

from scipy.stats import linregress

reg = linregress(arctic, antarctic)

print("Slope:", reg.slope)
print("Intercept:", reg.intercept)
print("R-squared:", reg.rvalue**2)


In [ ]:
# Generate a plot of the regression analysis

plt.figure(figsize=(8,6))
plt.scatter(arctic, antarctic, alpha=0.5, label="Data")

# Regression line
x_vals = np.linspace(arctic.min(), arctic.max(), 200)
y_vals = reg.intercept + reg.slope * x_vals
plt.plot(x_vals, y_vals, color="red", label="Regression Line")

plt.xlabel("Arctic Temperature Anomaly (°C)")
plt.ylabel("Antarctic Temperature Anomaly (°C)")
plt.title("Arctic vs Antarctic Temperature: Regression Analysis")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The OLS regression analysis yields a slope of 0.17, an intercept of -0.10, and an R-squared value of 0.054. The low R-squared indicates that Arctic temperature anomalies explain only about 5.4% of the variance in Antarctic temperature anomalies, further confirming the weak predictive relationship between the two regions.

In [ ]:
# Use Bayes Theorem and PyMC to sample the posterior probability distribution
# rho

x = df_clean["Arctic_Temp"].values
y = df_clean["Antarctic_Temp"].values

x_std = (x - x.mean()) / x.std()
y_std = (y - y.mean()) / y.std()

data = np.column_stack([x_std, y_std])

with pm.Model() as corr_model:
    # Prior on correlation
    rho = pm.Uniform("rho", lower=-1, upper=1)

    # Covariance matrix for standardized variables
    cov = pt.stack([
        [1.0, rho],
        [rho, 1.0]
    ])

    # Likelihood: bivariate normal
    pm.MvNormal("obs", mu=pt.zeros(2), cov=cov, observed=data)

    trace = pm.sample(2000, tune=2000, target_accept=0.9, random_seed=42)

az.summary(trace, var_names=["rho"])
az.plot_posterior(trace, var_names=["rho"], ref_val=0)


The posterior distribution of the correlation coefficient indicates that the most probable values are slightly positive, suggesting a weak positive relationship between Arctic and Antarctic temperature anomalies. Importantly, the 95% credible interval does not include zero, which provides evidence that the correlation is unlikely to be zero and therefore supports the presence of a real, though weak, relationship.

In [ ]:
!pip install bambi #system command to install bambi package

In [ ]:
import bambi as bmb #for automated Bayesian regression

In [ ]:
data = pd.DataFrame({"y": antarctic}) #define the y-variable
data["x_scaled"] = (arctic-np.mean(arctic)) / np.std(arctic) #scale the x values to have a mean of 0 and stdev of 1
data["x2_scaled"] = data["x_scaled"] ** 2 #used for quadratic regression
data["x3_scaled"] = data["x_scaled"] ** 3 #used for cubic regression
data["x4_scaled"] = data["x_scaled"] ** 4 #used for quartic regression

In [ ]:
# Fit a straight-line model (first-order polynomial)
model_linear = bmb.Model("y ~ x_scaled", data)
idata_linear = model_linear.fit(idata_kwargs={"log_likelihood": True})

# Fit a quadratic model (second-order polynomial)
model_quad = bmb.Model("y ~ x_scaled + x2_scaled", data)
idata_quad = model_quad.fit(idata_kwargs={"log_likelihood": True})

# Fit a cubic model (third-order polynomial)
model_cubic = bmb.Model("y ~ x_scaled + x2_scaled + x3_scaled", data)
idata_cubic= model_cubic.fit(idata_kwargs={"log_likelihood": True})

# Fit a quartic model (fourth-order polynomial)
model_quart = bmb.Model("y ~ x_scaled + x2_scaled + x3_scaled + x4_scaled", data)
idata_quart = model_quart.fit(idata_kwargs={"log_likelihood": True})

In [ ]:
# Compare models
comparison = az.compare({
    "linear": idata_linear,
    "quadratic": idata_quad,
    "cubic": idata_cubic,
    "quartic": idata_quart
})
print(comparison) #print the result

Model comparison using Leave-One-Out Cross-Validation (LOO-CV) shows that the quartic model achieves the highest elpd_loo score (-156.13), outperforming the cubic (-156.93), quadratic (-157.73), and linear (-168.21) models. This indicates that the quartic model provides the best balance between goodness-of-fit and generalization, and is therefore selected for subsequent predictive analysis.

In [ ]:
x_range = np.linspace(data["x_scaled"].min(), data["x_scaled"].max(), 100) # Predict across the range of scaled Arctic temperatures

# The new_data DataFrame needs to include all polynomial terms defined in model_quart
new_data = pd.DataFrame({
    "x_scaled": x_range,
    "x2_scaled": x_range**2,
    "x3_scaled": x_range**3,
    "x4_scaled": x_range**4
}) #dataframe with the new x-values

# Perform predictions for the mean and posterior predictive using model_quart
model_quart.predict(idata_quart, data=new_data, kind='response_params')
model_quart.predict(idata_quart, data=new_data, kind='response')

# Make random draws from the posterior of the regression lines (mean values, mu)
y_mean_draws = idata_quart.posterior["mu"].values

# Reshape to (total_draws, x_points)
y_mean_draws = y_mean_draws.reshape(-1, len(x_range))

# Make random draws from the posterior of the predicted values (posterior predictive)
y_pps_draws = idata_quart.posterior_predictive["y"].values

# Reshape to (total_draws, x_points)
y_pps_draws = y_pps_draws.reshape(-1, len(x_range))

# Compute mean and HDI of the posterior predictive distribution
posterior_mean = y_pps_draws.mean(axis=0)
hdi_mean = az.hdi(y_mean_draws, hdi_prob=0.95)
hdi_pps = az.hdi(y_pps_draws, hdi_prob=0.95)

# Plot both posterior for regression lines and the observations
plt.figure(figsize=(10, 6))
plt.plot(data["x_scaled"], data["y"], 'ok', label="Data")
plt.plot(x_range, posterior_mean, color="C1", label="Posterior predictive mean")
plt.fill_between(
    x_range,
    hdi_mean[:, 0],
    hdi_mean[:, 1],
    alpha=0.3,
    color="C1",
    label="95% HDI (mean)",
    edgecolor = None
)

plt.fill_between(
    x_range,
    hdi_pps[:, 0],
    hdi_pps[:, 1],
    alpha=0.2,
    color="C1",
    label="95% HDI (predictive)",
    edgecolor = None
)

plt.legend()
plt.minorticks_on()
plt.xlabel('Scaled Arctic Temperature Anomaly')
plt.ylabel('Antarctic Temperature Anomaly (°C)')
plt.title('Quartic Bayesian Regression: Scaled Arctic vs Antarctic Temperature Anomalies') # Updated title
plt.grid(alpha=0.3)
plt.show()

The quartic regression captures slight nonlinear trends in the relationship between Arctic and Antarctic temperature anomalies. However, the wide uncertainty bands (95% HDI), particularly in the predictive interval, indicate substantial variability and low confidence in precise predictions. This suggests that even with a more flexible model, Arctic temperatures provide limited predictive power for Antarctic temperatures, reinforcing the weak coupling between the two systems.

In [ ]:
year_data = df_clean["Year"]

# Set x_range based on the actual year data
x_range = np.linspace(year_data.min(), year_data.max(), 100)
new_data = pd.DataFrame({"Year": x_range}) # dataframe with the new x-values

# --- Calculate predictions for Arctic data from posterior coefficients ---
# Access posterior samples of intercept and slope from idata_arctic_year
intercept_samples_arctic = idata_arctic_year.posterior["Intercept"].values.flatten()
slope_samples_arctic = idata_arctic_year.posterior["Year"].values.flatten()

# Calculate arctic_y_mean_draws using broadcasting
arctic_y_mean_draws = intercept_samples_arctic[:, np.newaxis] + slope_samples_arctic[:, np.newaxis] * x_range

# Compute posterior mean and HDI at each x point
arctic_posterior_mean = arctic_y_mean_draws.mean(axis=0)
arctic_hdi = az.hdi(arctic_y_mean_draws, hdi_prob=0.95)

# --- Calculate predictions for Antarctic data from posterior coefficients ---
# Access posterior samples of intercept and slope from idata_antarctic_year
intercept_samples_antarctic = idata_antarctic_year.posterior["Intercept"].values.flatten()
slope_samples_antarctic = idata_antarctic_year.posterior["Year"].values.flatten()

# Calculate antarctic_y_mean_draws using broadcasting
antarctic_y_mean_draws = intercept_samples_antarctic[:, np.newaxis] + slope_samples_antarctic[:, np.newaxis] * x_range

# Compute posterior mean and HDI at each x point
antarctic_posterior_mean = antarctic_y_mean_draws.mean(axis=0)
antarctic_hdi = az.hdi(antarctic_y_mean_draws, hdi_prob=0.95)

# Plot the results
plt.figure(figsize=(12, 7))
plt.plot(year_data, df_clean["Arctic_Temp"], 'o', label="Arctic Data", color='firebrick', alpha=0.6)
plt.plot(year_data, df_clean["Antarctic_Temp"], 'o', label="Antarctic Data", color='royalblue', alpha=0.6)

# Arctic Regression Line and HDI
plt.plot(x_range, arctic_posterior_mean, color="firebrick", linestyle='--', label="Arctic Posterior Mean")
plt.fill_between(
    x_range,
    arctic_hdi[:, 0],
    arctic_hdi[:, 1],
    alpha=0.2,
    color="firebrick",
    label="Arctic 95% HDI",
    edgecolor = None
)

# Antarctic Regression Line and HDI
plt.plot(x_range, antarctic_posterior_mean, color="royalblue", linestyle='--', label="Antarctic Posterior Mean")
plt.fill_between(
    x_range,
    antarctic_hdi[:, 0],
    antarctic_hdi[:, 1],
    alpha=0.2,
    color="royalblue",
    label="Antarctic 95% HDI",
    edgecolor = None
)

plt.xlabel('Year')
plt.ylabel('Temperature Anomaly (°C)')
plt.title('Arctic vs Antarctic Temperature Anomalies with Bayesian Linear Regression')
plt.legend()
plt.minorticks_on()
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

Both regions show increasing temperature anomalies over time, confirming long-term warming trends. The Arctic regression line is steeper than that of the Antarctic, indicating a faster rate of warming. Overall, this supports the conclusion that Arctic warming is stronger and more consistent than Antarctic warming.

Our analysis reveals a real but weak positive correlation between Arctic and Antarctic temperature anomalies. According to the posterior predictive distribution, the mean correlation coefficient ρ is 0.22, with the 94% Highest Density Interval (HDI) residing entirely within the positive range (0.075 to 0.36). This provides strong evidence that the correlation is not merely a product of random noise.

As shown in the Bayesian Linear Regression figure above, the upward trends of both regression lines visually demonstrate that Arctic and Antarctic temperature anomalies are rising over time. Notably, the Arctic regression line is steeper than that of the Antarctic, indicating a faster rate of warming in the Arctic region.

Furthermore, as illustrated in the Quartic Bayesian Regression figure above, subtle non-linear relationships exist between the two polar regions’ temperature anomalies. However, the wide 95% HDI uncertainty bands suggest substantial variability, indicating that while a positive coupling exists, precise predictions remain limited.

These findings provide data-driven support for global climate feedback mechanisms, emphasizing that polar warming must be examined through a holistic, global system lens.

#Reflections
Kedi: For this week's project, I served as the Github & Integration lead. I focused on setting up the collaborative environment by importing the temple into Google Colab and managing access permissions to ensure smooth teamwork. I was responsible for troubleshooting technical hurdles and synchronizing our process back to the GitHub repository, maintaining an organized version history while facilitating efficient data sharing among team memebers.

Lee: My role in this project was to act as the data steward, I was responsible for finding and preparing the dataset used in the analysis. I focused on selecting a dataset that allowed for a comparison between Arctic and Antarctic warming and then cleaned and structured the data so it could be used for modelling. I learned that choosing the right dataset is not always straightforward. I initially looked at global temperature data but realised it did not contain the regional detail needed with a few data sets omitting polar temperatures.

Khai: It was my job to generate initial plots which involved the visualisation of data, linear regression and the probablility distribution of rho. Some of the concepts covered this week were a bit tricky to wrap my head around, but was very interesting to learn. I can see how they prove valuable and will be incorporating these methods in future reports.

Juliet: I created a visualisation of the deviations. Then, following Khai's probability distribution, I decided to make a polynomial regression model of the deviations, by first analysing which model was best. Following that I realised there was no proper comparision of time vs temperature for the arctic and Antarctic, so I made a linear regression model comparing the two. I then added interpretations for many of the outputs. In future, I could improve the comparative regression model so that it analyses the best model for the regression, and create predicitons from this. However, I did not have enough time to complete this additionally.

Binyao: My role in this project was to act as the narrative lead. I was responsible for writing the figure explanations and ensuring the written content accurately reflected the statistical results. This required me to develop a solid understanding of Bayesian methods, particularly how to interpret posterior distributions and HDI intervals, in order to translate the outputs into meaningful text. One challenge I faced was deciding which results needed further written explanation and which were sufficiently clear from the code and figures alone, so I could avoid repetition while maintaining a logical flow throughout the notebook.